# HDBSCAN 配對交易策略邏輯說明 (HDBSCAN Pairs Trading Strategy)

這份文件詳細說明了目前專案中所有基於 **HDBSCAN** 的配對交易策略邏輯。
這些策略主要分為 **形成期 (Formation Period)**、**交易期 (Trading Period)** 以及 **資金管理 (Capital Management)** 三個階段。

---

## 1. 策略概述
此策略參考了 Han et al. (2021) 的無監督學習配對交易框架，並將其系統解耦為獨立的形成期與交易期：
- **HDBSCAN (SS-PCA / SS-UMAP)**: 同產業內 (Same Sector) 分群。
- **HDBSCAN (CS-PCA / CS-UMAP / CS-MF)**: 跨產業 (Cross Sector) 分群，並搭配多種降維技術。
- **HDBSCAN (Macro-UMAP)**: 加入總體經濟指標進行跨產業分群。


## 2. 形成期 (Formation Period) 邏輯

形成期的目標是從大量的股票池中，利用無監督學習演算法挑選出具有高共整合性、高勝率的股票配對。

### 步驟 2.1: 特徵工程與降維 (Feature Engineering & Dimensionality Reduction)
根據不同的策略變體，我們對股票的歷史日報酬率進行處理：
- **PCA**: 使用主成分分析將報酬率降維，提取主要市場特徵。
- **UMAP**: 使用非線性降維技術 UMAP (Uniform Manifold Approximation and Projection)，更好地捕捉股票之間的非線性關係。
- **MF (Multi-Factor)**: 結合價格動能、波動率等多因子特徵。
- **Macro**: 結合總體經濟變數作為條件。

### 步驟 2.2: HDBSCAN 叢集分群 (Clustering)
- 將降維後的特徵輸入 **HDBSCAN** 演算法。
- 只有被分到同一個 Cluster 的股票才會被考慮作為潛在配對，被判定為 Noise (-1) 的股票將被排除。
- 設定 `min_cluster_size=30` 與 `min_samples=10` 等超參數以確保群集穩定性。

### 步驟 2.3: 潛在配對篩選與過濾 (Pair Filtering)
在同一群集中，兩兩股票形成潛在配對，並依序通過以下嚴格過濾：
1. **動量過濾 (Mom1 Filter)**: 
   - 確保兩檔股票在形成期的累積動量均大於特定門檻 (例如 `>= mom1_threshold`)。這有助於挑選出具有正向報酬潛力的強勢股配對，避免選入不斷下跌的弱勢股。
2. **皮爾森相關係數 (Pearson Correlation)**:
   - 兩檔股票價格的相關係數必須大於 `min_corr` (預設 0.50)，確保走勢具有基本一致性。
3. **共整合檢定 (Engle-Granger Cointegration Test / ADF Test)**:
   - 對配對的對數價格進行 OLS 迴歸: $\log(P_A) = lpha + eta \log(P_B) + \epsilon$
   - 提取殘差 (Spread)，並對殘差進行 ADF 單根檢定。
   - p-value 必須小於 `0.01`，證明殘差是穩定 (Stationary) 且具備均值回歸特性的。
4. **零軸穿越次數 (Zero Crossings)**:
   - 殘差穿越均值的次數必須大於 `min_zero_crossings` (預設 5 次)，確保配對不會長期偏離均值。

### 步驟 2.4: 參數儲存
最後，挑選出通過檢定的配對，記錄下 OLS 迴歸算出的 **Hedge_Ratio ($eta$)**、**OLS_Alpha ($lpha$)**、**Spread_Mean** 以及 **Spread_Std**，供交易期直接使用。


## 3. 交易期 (Trading Period) 邏輯

交易期讀取形成期產生的配對，在未來的測試期間內產生具體的買賣訊號，並計算 PnL。

### 步驟 3.1: 價差與 Z-Score 計算 (Spread & Z-Score)
每天根據最新的價格更新價差：
- $Spread_t = \log(Price_{A, t}) - (OLS\_Alpha + Hedge\_Ratio 	imes \log(Price_{B, t}))$
- $ZScore_t = rac{Spread_t - Spread\_Mean}{Spread\_Std}$
- *備註：解耦後的系統在開局前會額外往前抓取一段延伸期 (e.g. 20 天) 的資料，確保 Z-Score 滾動計算時不會產生 NaN，無縫接軌第一天的訊號。*

### 步驟 3.2: 交易訊號觸發 (Entry & Exit)
系統每天檢查 Z-Score 來決定是否進出場：
- **進場 (Entry)**: 
  - 若 $ZScore_t > 2.0$: 放空價差 (Enter Short A, Long B)。預期 A 被高估，B 被低估。
  - 若 $ZScore_t < -2.0$: 做多價差 (Enter Long A, Short B)。預期 A 被低估，B 被高估。
- **出場 (Exit)**:
  - 若目前做空價差，且 $ZScore_t \leq 0.0$：平倉獲利了結。
  - 若目前做多價差，且 $ZScore_t \geq 0.0$：平倉獲利了結。

### 步驟 3.3: 停損機制 (Stop Loss)
- **資金停損 (Capital Stop Loss)**:
  - 若單筆交易的未實現虧損超過配對分配資金的給定比例 (如 `stop_loss_pct = 0.05` 即 5%)，則強制停損出場。
- **動態停損 (Dynamic Stop Loss)**:
  - (目前設定為 **False**) 由於我們使用靜態的 OLS 參數，若開啟不當的 Z-Score 動態停損 (例如大於 0 即停損)，會導致一進場立刻被洗出的致命 Bug。目前依據原邏輯關閉此設定，放任價差回歸。


## 4. 資金管理 (Capital Management / Portfolio Manager)

為貼近真實交易情境，系統內建了一個 `PortfolioManager` 來管理整體帳戶資金與倉位：

### 步驟 4.1: 資金分配 (Capital Allocation)
- **固定金額分配**: 在每期交易開始時，將總資金 (預設 `$10000`) 平均分配給排名前 $N$ (Top N) 的配對。
- 例如：若設定 `Top N = 5`，則每一對配對將獲得 $10000 / 5 = 2000$ 的初始操作資金。
- 這筆資金決定了該配對能購買多少股數 (Shares)。由於包含做空與做多，需計算槓桿並扣除進場時的手續費與滑價 (預設 `0.1%` fee + `0.1%` slippage)。

### 步驟 4.2: 最大產業集中度 (Max Sector Ratio)
- 為了避免將資金過度集中於單一產業，資金管理模組支援 `max_sector_ratio` (MSR) 的設定。
- 若 `MSR = 0.3` (30%)，則單一產業最多只能佔總資金的 30%。當某一產業的配對數超過限制時，多餘的配對將被剔除，並由其他產業的候補配對遞補。

### 步驟 4.3: 總體績效計算 (Performance Tracking)
- `PortfolioManager` 每天追蹤所有配對的 **未實現損益 (Unrealized PnL)** 與 **已實現損益 (Realized PnL)**。
- 交易結束時計算總體的：
  - **累計報酬率 (Cum_Ret)**
  - **年化報酬率 (Ann_Ret)**
  - **最大回撤 (Max Drawdown)**
  - **夏普值 (Sharpe Ratio)**
- *注意：目前資金管理採取較保守的無複利 (No Compounding) 配置，各期的虧損會反映在總資金下降，但每期的分配資金仍以期初為準進行等比例拆分。*
